In [1]:
# import urllib.request
# import ssl

# url = "https://images.cocodataset.org/train2017/000000301684.jpg"

# ssl._create_default_https_context = ssl._create_unverified_context

# try:
#     urllib.request.urlretrieve(url, "000000301684.jpg")
#     print("Image downloaded successfully.")
# except urllib.error.URLError as e:
#     print(f"Error downloading image: {e}")

# from PIL import Image
# try:
#     img = Image.open("000000301684.jpg")
#     img.show()
# except FileNotFoundError:
#     print("Downloaded image file not found.")

## **Disclaimer:**
As mentioned in the paper the inference is quite expensive, running this locally was impractical for me. For this reason I tryed this workaround, creating a notebook that I would have been able to run on Google Colab, forking the repository inserting this notebook and doing some changes to the paths on the python files to make everything work in Google Colab.

This notebook contains a slightly modified version of uni-ddim-inversion.py and sim.py.

# Required libraries and imports

In [2]:
!git clone https://github.com/MarkBridge11/ZeroFake-Mod.git

Cloning into 'ZeroFake-Mod'...
remote: Enumerating objects: 79, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 79 (delta 26), reused 36 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (79/79), 205.88 KiB | 9.36 MiB/s, done.
Resolving deltas: 100% (26/26), done.


In [3]:
import sys
sys.path.append('/content/ZeroFake-Mod')

In [4]:
!pip install fairscale timm natsort scikit-image einops kornia spacy albumentations pudb invisible-watermark imageio imageio-ffmpeg omegaconf test-tube streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 5.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [5]:
from functools import partial
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple, Union

import numpy as np
import PIL
import torch
import torch.nn.functional as nnf

from diffusers import DDIMScheduler, StableDiffusionPipeline

path = "/home/c01zesh/CISPA-projects/meta_transfer-2023/stable-diffusion/stable-diffusion-v1-4"
path = Path(path).expanduser()
# These variable is not used in my case because here they were importing locally stable-diffusion model. The path should correspond to some path in Linux system

from PIL import Image
from torchvision import transforms

import inspect

import torch
from transformers import CLIPFeatureExtractor, CLIPTextModel, CLIPTokenizer

from diffusers.configuration_utils import FrozenDict
from diffusers.models import AutoencoderKL, UNet2DConditionModel
try:
    from diffusers.pipeline_utils import DiffusionPipeline
except:
    from diffusers.pipelines.pipeline_utils import DiffusionPipeline
from diffusers.pipelines.stable_diffusion import StableDiffusionPipelineOutput
from diffusers.pipelines.stable_diffusion.safety_checker import \
    StableDiffusionSafetyChecker
from diffusers.schedulers import DDIMScheduler,PNDMScheduler, LMSDiscreteScheduler
from diffusers.utils import deprecate, logging, numpy_to_pil
from natsort import natsorted

from PIL import Image
import requests
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode
from blipmodels import blip_decoder
import spacy

import cv2
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as PSNR
from skimage.metrics import mean_squared_error as MSE

import os
import sys
import argparse

# Helper functions

In [6]:
def replace_first_noun(sentence, replacement):
    doc = nlp(sentence)
    for token in doc:

        if token.pos_ == "NOUN":
            return sentence[:token.idx] + replacement + sentence[token.idx + len(token.text):]

    return sentence

def backward_ddim(x_t, alpha_t: float, alpha_tm1: float, eps_xt):
    """ from noise to image"""
    return (
        alpha_tm1**0.5
        * (
            (alpha_t**-0.5 - alpha_tm1**-0.5) * x_t
            + ((1 / alpha_tm1 - 1) ** 0.5 - (1 / alpha_t - 1) ** 0.5) * eps_xt
        )
        + x_t
    )

def forward_ddim(x_t, alpha_t: float, alpha_tp1: float, eps_xt):
    """ from image to noise, it's the same as backward_ddim"""
    return backward_ddim(x_t, alpha_t, alpha_tp1, eps_xt)

def load_img(path, target_size=512):
    """Load an image, resize and output -1..1"""
    image = Image.open(path).convert("RGB")

    tform = transforms.Compose(
        [
            transforms.Resize((target_size,target_size)), #resizing to 512x512
            transforms.CenterCrop(target_size), #center cropping
            transforms.ToTensor(),
        ]
    )
    image = tform(image)
    return 2.0 * image - 1.0 # converting the image into [-1,1] range

# Redifinition of StableDiffusionPipeline

In [7]:
class StableDiffusionPipeline(DiffusionPipeline):
    def __init__(
        self,
        vae: AutoencoderKL,
        text_encoder: CLIPTextModel,
        tokenizer: CLIPTokenizer,
        unet: UNet2DConditionModel,
        scheduler: Union[DDIMScheduler, PNDMScheduler, LMSDiscreteScheduler],
        #safety_checker: StableDiffusionSafetyChecker = None,
        #feature_extractor: CLIPFeatureExtractor = None
    ):
        super().__init__()

        self.register_modules( # registers the following components in the pipeline
            vae=vae,
            text_encoder=text_encoder,
            tokenizer=tokenizer,
            unet=unet,
            scheduler=scheduler,
        )
        self.forward_diffusion = partial(self.backward_diffusion, reverse_process=True)

    """
    Transform the prompt into text embedding using CLIPTokenizer
    """
    @torch.inference_mode()
    def get_text_embedding(self, prompt):
        text_input_ids = self.tokenizer(
            prompt,
            padding="max_length",
            truncation=True,
            max_length=self.tokenizer.model_max_length,
            return_tensors="pt",
        ).input_ids
        text_embeddings = self.text_encoder(text_input_ids.to(self.device))[0]
        return text_embeddings


    """
    We obtain the latent vector of the input image of the VAE.
    First we retrieve the DiagonalGaussianDistribution (encoding_dist) that it uses it to sample from it or directly from the mean using .mode().
    Then the latents are scaled.
    """
    @torch.inference_mode()
    def get_image_latents(self, image, sample=True, rng_generator=None):
        encoding_dist = self.vae.encode(image).latent_dist # as mentioned in the paper it's a zero-shot approach but we still need to know a VAE and UNet of a LDM
        if sample:
            encoding = encoding_dist.sample(generator=rng_generator)
        else:
            encoding = encoding_dist.mode()
        latents = encoding * 0.18215
        return latents

    """
    Despite the name this function can do both backward and forward DDIM sampling. In our case it is used as inversion and reconstruction exploiting formulas defined above.
    """
    @torch.inference_mode()
    def backward_diffusion(
        self,
        use_old_emb_i=25, # set the inference step from which changes the text embedding to the modified one
        text_embeddings=None,
        old_text_embeddings=None, # not modified prompt
        new_text_embeddings=None, # modified prompt
        latents: Optional[torch.FloatTensor] = None, # initial latent input, x0 if reverse, x_T if reconstruction
        num_inference_steps: int = 50,
        guidance_scale: float = 7.5, # strength of classifier-free guidance
        callback: Optional[Callable[[int, int, torch.FloatTensor], None]] = None,
        callback_steps: Optional[int] = 1,
        reverse_process: True = False,
        **kwargs,
    ):
        """
        Generate image from text prompt and latents
        """
        do_classifier_free_guidance = guidance_scale > 1.0 # enables classifier-free guidance if guidance scale > 1.0
        self.scheduler.set_timesteps(num_inference_steps)
        # initialize values of alphas, based on the number of inference steps, so we separate 1k training steps based on num_inference_steps
        # 1k, if we have 50 as inference steps, we will have [980, 960, ..., 0] for example
        timesteps_tensor = self.scheduler.timesteps.to(self.device)
        latents = latents * self.scheduler.init_noise_sigma
        # scales the input latents to the same noise level on which the model was trained on
        # -> because VAE scales to a distribution, but might not be the same used in training, so we take it from the scheduler to be sure

        if old_text_embeddings is not None and new_text_embeddings is not None: # a bool to be sure to use the modified prompt from a certain inference step
            prompt_to_prompt = True
        else:
            prompt_to_prompt = False


        for i, t in enumerate(self.progress_bar(timesteps_tensor if not reverse_process else reversed(timesteps_tensor))):
        # this is to show a progress bar when we do inversion process
            if prompt_to_prompt:
                if i < use_old_emb_i: # here it replaces the old one with the modified one
                    text_embeddings = old_text_embeddings
                else:
                    text_embeddings = new_text_embeddings

            latent_model_input = (
                torch.cat([latents] * 2) if do_classifier_free_guidance else latents # if we're using cfg, it duplicates the latents
            )
            latent_model_input = self.scheduler.scale_model_input(latent_model_input, t)
            # So it scales the obtained latent based on the timestep. We must match the scale of the latents used during training, otherwise we would obtain garbage predictions
            noise_pred = self.unet(
                latent_model_input, t, encoder_hidden_states=text_embeddings #predicting the noise guided by text embedding
            ).sample

            if do_classifier_free_guidance:
                noise_pred_uncond, noise_pred_text = noise_pred.chunk(2) # it splits the noise prediction into two parts, one for uncod and the other cond
                noise_pred = noise_pred_uncond + guidance_scale * (
                    noise_pred_text - noise_pred_uncond
                )

            prev_timestep = ( # calculate the previous timestep index for DDIM update
                t
                - self.scheduler.config.num_train_timesteps
                // self.scheduler.num_inference_steps
            )
            # the ratio gives the interval between steps in order to retrieve the previous timestep as a subtraction with the current t.
            # num_train_timeteps is the number of diffusion steps used during training of the diffusion model (typically 1k), that defines how gradually noise is added in forward process.
            # During inference you try to do 1k in 50 inference_steps instead. So those 50 steps are spaced within the 1k-step training schedule

            if callback is not None and i % callback_steps == 0:
                callback(i, t, latents)

            # Here we compute alpha_t and alpha_t-1 (this latter one based on prev_timestep)
            alpha_prod_t = self.scheduler.alphas_cumprod[t] # this is the overline_alpha_t, the cumulative product of alphas up to time t
            alpha_prod_t_prev = ( # we calculate overline_alpha_t-1, setting it to the previous timestep (an approximation of what it was)
                self.scheduler.alphas_cumprod[prev_timestep]
                if prev_timestep >= 0
                else self.scheduler.final_alpha_cumprod # or directly to the last one
            )
            if reverse_process: # if we're doing the reverse process we will swap the directions, inverting variable values
                alpha_prod_t, alpha_prod_t_prev = alpha_prod_t_prev, alpha_prod_t
            latents = backward_ddim( # perform reverse/reconstruction process
                x_t=latents,
                alpha_t=alpha_prod_t,
                alpha_tm1=alpha_prod_t_prev,
                eps_xt=noise_pred,
            )
        return latents


    @torch.inference_mode()
    def decode_image(self, latents: torch.FloatTensor, **kwargs) -> List[Image.Image]:
        scaled_latents = 1 / 0.18215 * latents
        image = [
            self.vae.decode(scaled_latents[i : i + 1]).sample for i in range(len(latents))
        ]
        image = torch.cat(image, dim=0)
        return image

    @torch.inference_mode()
    def torch_to_numpy(self, image) -> List[Image.Image]: # this function was missing and giving error
        image = (image / 2 + 0.5).clamp(0, 1)
        image = image.cpu().permute(0, 2, 3, 1).numpy()
        return image

In [8]:
pipe = StableDiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4")
#pipe.scheduler = DDIMScheduler.from_config(path / "scheduler")
# They were importing locally the scheduler. However it is a zero-shot approach so this should not be something that they have trained and that lead to some perfomance changes.
# The pipe already include its scheduler, so there's not point on importing it.
pipe = pipe.to("cuda")

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

The config attributes {'feature_extractor': ['transformers', 'CLIPImageProcessor'], 'safety_checker': ['stable_diffusion', 'StableDiffusionSafetyChecker']} were passed to StableDiffusionPipeline, but are not expected and will be ignored. Please verify your model_index.json configuration file.
Keyword arguments {'feature_extractor': ['transformers', 'CLIPImageProcessor'], 'safety_checker': ['stable_diffusion', 'StableDiffusionSafetyChecker']} are not expected by StableDiffusionPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

# Image folder creation

In [9]:
# parser = argparse.ArgumentParser(description="Process images for DDIM.")
# parser.add_argument("--target", type=str, help="Path to the folder containing images.")
# parser.add_argument("--output", type=str, help="Path to the output folder for saving images.")
# args = parser.parse_args()

output_folder = Path("/content/ZeroFake-Mod/imgs")
# output_folder = args.output
# os.makedirs(output_folder, exist_ok=True)

In [10]:
images_path = Path("/content/ZeroFake-Mod/imgs")
images_list = list(images_path.glob("*.jpg")) + \
              list(images_path.glob("*.jpeg")) + \
              list(images_path.glob("*.png"))

images_list_sorted = natsorted(images_list) # Sorting like a human
images_list_str= [str(x) for x in images_list_sorted]

# Loading BLIP model to generate image prompt

In [11]:
index = 0
nlp = spacy.load("en_core_web_sm")
model_url = 'https://storage.googleapis.com/sfr-vision-language-research/BLIP/models/model_base_capfilt_large.pth'
image_size = 512
print("Loading BLIP model...")
blipmodel = blip_decoder(pretrained=model_url, image_size=image_size, vit='base') # Error solved when using generate with this, that can be found in blip.py and med.py
print("BLIP model loaded.")
blipmodel.eval()
blipmodel = blipmodel.cuda()

print("Found", len(images_list_str), "images.")
print("images_list_str =", images_list_str)

Loading BLIP model...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

100%|██████████| 1.97G/1.97G [00:16<00:00, 131MB/s]


reshape position embedding from 196 to 1024
load checkpoint from https://storage.googleapis.com/sfr-vision-language-research/BLIP/models/model_base_capfilt_large.pth
BLIP model loaded.
Found 1 images.
images_list_str = ['/content/ZeroFake-Mod/imgs/ann000000005136.png']


In [12]:
# print(type(blipmodel))                     # Should print <class 'blipmodels.blip.BLIP_Decoder'>
# print(blipmodel.__class__.__module__)     # Should print 'blipmodels.blip'

# import inspect
# print(inspect.getfile(blip_decoder))  # Should show '/content/ZeroFake-Mod/blipmodels/blip.py'

# print(blipmodel)
# print(type(blipmodel.text_decoder))
# print(hasattr(blipmodel.text_decoder, "generate"))  # Should be True if valid

# DDIM Inversion and reconstruction

This code chunk represent the core of the approach, it is calling the manually defined functions of inversion and reconstruction. In fact it is not calling an instantiation of the pipeline, but using methods that we have defined such as forward diffusion and backward diffusion. Then we manually reconstruct from latents the image, that will be later used to understand the results.

In [13]:
for impath in images_list_str:
    print("Entering in for loop")
    img = load_img(impath).unsqueeze(0).to("cuda")
    print(index)

    prompt = blipmodel.generate(img, sample=True, num_beams=3, max_length=40, min_length=5)[0]

    print(prompt)

    prompt = replace_first_noun(prompt, 'big tree') # using big tree as replacement for every input is not optimal

    print(prompt)

    text_embeddings = pipe.get_text_embedding(prompt)

    rng_generator=torch.Generator(device=pipe.device).manual_seed(0)

    image_latents = pipe.get_image_latents(img, rng_generator)

    reversed_latents = pipe.forward_diffusion(
        latents=image_latents,
        text_embeddings=text_embeddings,
        guidance_scale=1,
        num_inference_steps=999,
    )

    reconstructed_latents = pipe.backward_diffusion(
        latents=reversed_latents,
        text_embeddings=text_embeddings,
        guidance_scale=1,
        num_inference_steps=20,
    )

    # guidance_scale=1 so we follow the prompt only

    def latents_to_imgs(latents):
        x = pipe.decode_image(latents)
        x = pipe.torch_to_numpy(x)
        x = numpy_to_pil(x)
        return x

    image = latents_to_imgs(reconstructed_latents)[0]

    print(f"Saving image {index} to {os.path.join(output_folder, str(index) + '.png')}")
    image.save(os.path.join(output_folder, str(index) + '.png'), format="PNG")
    index += 1

Entering in for loop
0
elephants in an open area
big tree in an open area


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/21 [00:00<?, ?it/s]

Saving image 0 to /content/ZeroFake-Mod/imgs/0.png


# Analyzing the results

In [14]:
# parser = argparse.ArgumentParser(description="Process images for DDIM.")
# parser.add_argument("--orginal", type=str, help="Path to the folder containing images.")
# parser.add_argument("--reconstruct", type=str, help="Path to the output folder for saving images.")
# args = parser.parse_args()


# folder1_path = args.orginal

# folder2_path = args.reconstruct

output_file = "testfake.txt"

with open(output_file, "w") as f:
    f.write("Image Filename\tPixel Similarity\n")
    #index = 0
    #for image_path1, image_path2 in zip(natsorted(Path(folder1_path).glob("*.*")), natsorted(Path(folder2_path).glob("*.*"))):

    image1 = cv2.imread(str("/content/ZeroFake-Mod/imgs/ann000000005136.png"))
    image2 = cv2.imread(str("/content/ZeroFake-Mod/imgs/0.png"))
    image1 = cv2.resize(image1, (512, 512))
    image2 = cv2.resize(image2, (512, 512))

    gray1 = cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY)
    ssim_score = ssim(gray1, gray2)
    f.write(f"Fake img\t{ssim_score}\n")
    print("Mean pixel difference for fake image:", ssim_score)
    index+=1

    print("Results saved to", output_file)

Mean pixel difference for fake image: 0.78430788123864
Results saved to testfake.txt
